In [ ]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.
import kagglehub
mengcius_cinic10_path = kagglehub.dataset_download('mengcius/cinic10')

print('Data source import complete.')
from google.colab import drive
drive.mount('/content/drive')

Using Colab cache for faster access to the 'cinic10' dataset.
Data source import complete.
Mounted at /content/drive


## Imports

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import os, time, csv, shutil
import pandas as pd
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import ConcatDataset, DataLoader
from sklearn.metrics import f1_score
import random
import time
from torch.utils.data import TensorDataset
from torchvision.transforms import v2
from sklearn.metrics import f1_score
from torch.utils.data.dataloader import default_collate

In [ ]:
cinic_mean_RGB = [0.47889522, 0.47227842, 0.43047404]
cinic_std_RGB  = [0.24205776, 0.23828046, 0.25874835]

SEEDS      = [42, 123, 2024, 7, 999]
DATA_PATH = mengcius_cinic10_path
# DATA_PATH  = '/kaggle/input/cinic10/'
# DATA_PATH = "/kaggle/working/dataset"
# if not os.path.exists(DATA_PATH):
#     shutil.copytree(src, DATA_PATH)
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print(f"Using device: {DEVICE}")

Using device: cuda


In [ ]:
# shutil.rmtree("/kaggle/working/experiments")

## Ensuring reproducibility

In [ ]:
def set_seed(seed: int):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

## Data Loaders

In [ ]:
base_transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
])

image_aug_pipelines = {
    'flip': transforms.Compose([
        transforms.RandomHorizontalFlip(p=1.0),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'blur': transforms.Compose([
        transforms.GaussianBlur(kernel_size=3, sigma=(0.1, 2.0)),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'jitter': transforms.Compose([
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.1),
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutout': transforms.Compose([
        transforms.ToTensor(),
        transforms.RandomErasing(p=1.0, scale=(0.25, 0.25), ratio=(1, 1), value=0),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha_1': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ]),
    'cutmix_alpha_4': transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize(cinic_mean_RGB, cinic_std_RGB)
    ])
}

batch_aug_pipelines = {
    'cutmix_alpha_1': transforms.v2.CutMix(num_classes=10, alpha=1.0),
    'cutmix_alpha_4': transforms.v2.CutMix(num_classes=10, alpha=4.0)
}

In [ ]:
def preload_to_ram(dataset):
    loader = DataLoader(dataset, batch_size=512, num_workers=4, pin_memory=False)
    all_images, all_labels = [], []
    for images, labels in loader:
        all_images.append(images)
        all_labels.append(labels)
    return TensorDataset(torch.cat(all_images), torch.cat(all_labels))

In [ ]:
from torch.utils.data import default_collate
def train_collate_fn(aug_type, p=1):
    def collate_fn(batch):
        cutmix = batch_aug_pipelines[aug_type]
        images, labels = default_collate(batch)
        if torch.rand(()) < p:
            images, labels = cutmix(images, labels)
        return images, labels
    return collate_fn

In [ ]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, ConcatDataset

_loader_cache = {}

def get_loaders(batch_size: int, path: str = DATA_PATH, aug_type: str = None, num_workers: int = 4):
    key = (path, batch_size, aug_type, num_workers)

    if key in _loader_cache:
        return _loader_cache[key]

    pin_memory = torch.cuda.is_available()

    train_base = datasets.ImageFolder(
        root=os.path.join(path, "train"),
        transform=base_transform,
    )
    valid_ds = datasets.ImageFolder(
        root=os.path.join(path, "valid"),
        transform=base_transform,
    )
    test_ds = datasets.ImageFolder(
        root=os.path.join(path, "test"),
        transform=base_transform,
    )

    valid_ds = preload_to_ram(valid_ds)
    test_ds = preload_to_ram(test_ds)

    train_eval_loader = DataLoader(
        train_base,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    train_size_for_log = None

    if aug_type is None:
        train_ds = preload_to_ram(train_base)

        train_loader = DataLoader(
            train_ds,
            batch_size=batch_size,
            shuffle=True,
            num_workers=num_workers,
            pin_memory=pin_memory,
            persistent_workers=(num_workers > 0),
        )

        train_size_for_log = len(train_ds)

    else:
        if aug_type in ("cutmix_alpha_1", "cutmix_alpha_4"):
            train_clean_ds = preload_to_ram(train_base)
            train_cutmix_ds = preload_to_ram(train_base)

            train_clean_loader = DataLoader(
                train_clean_ds,
                batch_size=batch_size,
                shuffle=True,
                num_workers=num_workers,
                pin_memory=pin_memory,
                persistent_workers=(num_workers > 0),
            )

            train_cutmix_loader = DataLoader(
                train_cutmix_ds,
                batch_size=batch_size,
                shuffle=True,
                num_workers=num_workers,
                pin_memory=pin_memory,
                persistent_workers=(num_workers > 0),
                collate_fn=train_collate_fn(aug_type, p=1.0),
            )

            train_loader = (train_clean_loader, train_cutmix_loader)
            train_size_for_log = len(train_clean_ds) + len(train_cutmix_ds)

        else:
            if aug_type == "random":
                standard_transforms = {k: v for k, v in image_aug_pipelines.items()
                                       if k not in ('cutmix_alpha_1', 'cutmix_alpha_4')}
                chosen_transform = transforms.RandomChoice(list(standard_transforms.values()))
            else:
                chosen_transform = image_aug_pipelines[aug_type]

            train_aug = datasets.ImageFolder(
                root=os.path.join(path, "train"),
                transform=chosen_transform,
            )

            train_ds = ConcatDataset([train_base, train_aug])

            train_loader = DataLoader(
                train_ds,
                batch_size=batch_size,
                shuffle=True,
                num_workers=num_workers,
                pin_memory=pin_memory,
                persistent_workers=(num_workers > 0),
            )

            train_size_for_log = len(train_ds)

    valid_loader = DataLoader(
        valid_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    test_loader = DataLoader(
        test_ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=pin_memory,
        persistent_workers=(num_workers > 0),
    )

    print(
        f"len(train)={train_size_for_log} | "
        f"len(valid_ds)={len(valid_ds)} | "
        f"len(test_ds)={len(test_ds)}"
    )

    _loader_cache[key] = (train_loader, train_eval_loader, valid_loader, test_loader)
    return _loader_cache[key]

In [ ]:
# train, valid, test = get_loaders(batch_size = 64, aug_type = "cutmix")

In [ ]:
# import torchvision
# import matplotlib.pyplot as plt
# from torch.utils.data import default_collate

# mean = torch.tensor(cinic_mean_RGB).view(3,1,1)
# std  = torch.tensor(cinic_std_RGB).view(3,1,1)

# def denorm_batch(x):
#     return (x * std + mean).clamp(0, 1)

# images, labels = next(iter(train))

# imgs = denorm_batch(images.cpu())
# grid = torchvision.utils.make_grid(imgs, nrow=4)

# plt.figure(figsize=(40, 15))
# plt.imshow(grid.permute(1,2,0))
# plt.axis("off")
# plt.show()

## Ensuring mobilenetv2 availability

In [ ]:
def get_mobilenetv2(num_classes, dropout_rate = 0):
    m = models.mobilenet_v2()
    old_conv = m.features[0][0]
    m.features[0][0] = nn.Conv2d(
        in_channels=old_conv.in_channels,
        out_channels=old_conv.out_channels,
        kernel_size=old_conv.kernel_size,
        stride=1,
        padding=old_conv.padding,
        dilation=old_conv.dilation,
        groups=old_conv.groups,
        bias=(old_conv.bias is not None)
    )
    m.classifier = nn.Sequential(
        nn.Dropout(p=dropout_rate),
        nn.Linear(m.classifier[1].in_features, num_classes)
    )
    return m

# Base function for experiments

In [ ]:
import os
import csv
import time
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.metrics import f1_score

def train_one_sample(cfg, num_classes=10, out_root="/content/drive/MyDrive/advanced_random"):
    set_seed(cfg.get("seed", 42))

    run_dir = os.path.join(out_root, cfg["run_name"])
    os.makedirs(run_dir, exist_ok=True)

    print(f"Starting run: {cfg['run_name']} with config: {cfg}")
    print(range(cfg.get("epochs", 25) + 1))

    train_loader, train_eval_loader, valid_loader, test_loader = get_loaders(
        batch_size=cfg["batch_size"],
        aug_type=cfg["aug"]
    )

    print(
        f"Data loaders ready. "
        f"Train batches: {len(train_loader)}, "
        f"Train-eval batches: {len(train_eval_loader)}, "
        f"Valid batches: {len(valid_loader)}, "
        f"Test batches: {len(test_loader)}"
    )

    model = get_mobilenetv2(
        num_classes=num_classes,
        dropout_rate=cfg["dropout"]
    ).to(DEVICE)

    print(model)

    optimizer = optim.Adam(
        model.parameters(),
        lr=cfg["lr"],
        weight_decay=cfg["weight_decay"]
    )

    criterion = nn.CrossEntropyLoss()

    print("lr:", cfg["lr"])
    print("batch_size:", cfg["batch_size"])
    print("weight_decay:", cfg["weight_decay"])
    print("dropout:", cfg["dropout"])
    print("aug:", cfg["aug"])

    metrics_csv = os.path.join(run_dir, "metrics.csv")
    if not os.path.exists(metrics_csv):
        with open(metrics_csv, "w", newline="") as f:
            writer = csv.writer(f)
            writer.writerow([
                "epoch",
                "train_loss",
                "val_loss",
                "train_f1_macro",
                "val_f1_macro",
                "lr",
                "batch_size"
            ])

    best_val_f1 = 0.0
    last_checkpoint_time = time.time()
    checkpoint_duration = cfg.get("checkpoint_duration", 5) * 60
    step = 0

    def save_checkpoint(name, epoch):
        torch.save({
            "epoch": epoch,
            "step": step,
            "best_f1": best_val_f1,
            "model": model.state_dict(),
            "opt": optimizer.state_dict(),
            "cfg": cfg
        }, os.path.join(run_dir, f"{name}.pth"))

    epochs = cfg.get("epochs", 25)
    print(f"Starting training for {epochs} epochs...")

    for epoch in range(epochs + 1):
        model.train()

        training_sum = 0.0
        training_n = 0
        if cfg["aug"] in ("cutmix_alpha_1", "cutmix_alpha_4"):
          train_clean_loader, train_cutmix_loader = train_loader

          for loader in (train_clean_loader, train_cutmix_loader):
              for x, y in loader:
                  x = x.to(DEVICE, non_blocking=True)
                  y = y.to(DEVICE, non_blocking=True)

                  optimizer.zero_grad(set_to_none=True)
                  logits = model(x)
                  loss = criterion(logits, y)
                  loss.backward()
                  optimizer.step()

                  step += 1

                  batch_size_now = x.size(0)
                  training_sum += loss.item() * batch_size_now
                  training_n += batch_size_now

                  if time.time() - last_checkpoint_time > checkpoint_duration:
                    save_checkpoint(name="checkpoint", epoch=epoch)
                    last_checkpoint_time = time.time()
        else:
          for x, y in train_loader:
              x = x.to(DEVICE, non_blocking=True)
              y = y.to(DEVICE, non_blocking=True)

              optimizer.zero_grad(set_to_none=True)
              logits = model(x)
              loss = criterion(logits, y)
              loss.backward()
              optimizer.step()

              step += 1

              batch_size_now = x.size(0)
              training_sum += loss.item() * batch_size_now
              training_n += batch_size_now

              if time.time() - last_checkpoint_time > checkpoint_duration:
                save_checkpoint(name="checkpoint", epoch=epoch)
                last_checkpoint_time = time.time()

        # for x, y in train_loader:
        #     x = x.to(DEVICE, non_blocking=True)
        #     y = y.to(DEVICE, non_blocking=True)

        #     optimizer.zero_grad(set_to_none=True)

        #     logits = model(x)
        #     loss = criterion(logits, y)
        #     loss.backward()
        #     optimizer.step()

        #     step += 1

        #     batch_size_now = x.size(0)
        #     training_sum += loss.item() * batch_size_now
        #     training_n += batch_size_now

        #     if time.time() - last_checkpoint_time > checkpoint_duration:
        #         save_checkpoint(name="checkpoint", epoch=epoch)
        #         last_checkpoint_time = time.time()

        train_loss = training_sum / max(training_n, 1)

        # ===== EVAL =====
        model.eval()

        train_preds_all, train_labels_all = [], []
        valid_preds_all, valid_labels_all = [], []

        valid_loss_sum = 0.0
        valid_n = 0

        with torch.inference_mode():
            # train metrics liczone na czystym train_eval_loader
            for x, y in train_eval_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)

                logits = model(x)

                train_preds_all.append(logits.argmax(dim=1).cpu())
                train_labels_all.append(y.cpu())

            # validation metrics
            for x, y in valid_loader:
                x = x.to(DEVICE, non_blocking=True)
                y = y.to(DEVICE, non_blocking=True)

                logits = model(x)
                loss = criterion(logits, y)

                valid_loss_sum += loss.item() * x.size(0)
                valid_n += x.size(0)

                valid_preds_all.append(logits.argmax(dim=1).cpu())
                valid_labels_all.append(y.cpu())

        train_preds_all = torch.cat(train_preds_all).numpy()
        train_labels_all = torch.cat(train_labels_all).numpy()
        train_f1_macro = f1_score(train_labels_all, train_preds_all, average="macro")

        valid_preds_all = torch.cat(valid_preds_all).numpy()
        valid_labels_all = torch.cat(valid_labels_all).numpy()
        val_f1_macro = f1_score(valid_labels_all, valid_preds_all, average="macro")

        val_loss = valid_loss_sum / max(valid_n, 1)
        lr_now = optimizer.param_groups[0]["lr"]

        with open(metrics_csv, "a", newline="") as f:
            csv.writer(f).writerow([
                epoch,
                train_loss,
                val_loss,
                train_f1_macro,
                val_f1_macro,
                lr_now,
                cfg["batch_size"]
            ])

        save_checkpoint(name="last", epoch=epoch)

        if val_f1_macro > best_val_f1:
            best_val_f1 = val_f1_macro
            save_checkpoint(name="best", epoch=epoch)

        print(
            f'[{cfg["run_name"]}] '
            f'{epoch:02d}/{epochs-1} '
            f'/ bs={cfg["batch_size"]} lr={lr_now:g} '
            f'/ train={train_loss:.4f} val={val_loss:.4f} '
            f'train_f1={train_f1_macro:.4f} val_f1={val_f1_macro:.4f} '
            f'best={best_val_f1:.4f}'
        )

    torch.save(model.state_dict(), os.path.join(run_dir, "final_weights.pth"))
    return run_dir, best_val_f1

# Learning rate experiments

In [ ]:
# experiments_lr = []

# for seed in SEEDS:
#     experiments_lr.append({"run_name":f"mbv2_bs64_lr1e-2_seed_{seed}", "batch_size":64, "lr":1e-2, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_lr.append({"run_name":f"mbv2_bs64_lr1e-3_seed_{seed}", "batch_size":64, "lr":1e-3, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_lr.append({"run_name":f"mbv2_bs64_lr1e-4_seed_{seed}", "batch_size":64, "lr":1e-4, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})

# print(experiments_lr)

# results = []
# for cfg in experiments_lr:
#     run_dir, best_f1 = train_one_sample(cfg)
#     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

In [ ]:
# experiments_lr = []

# for seed in SEEDS[-3:]:
#     experiments_lr.append({"run_name":f"mbv2_bs64_lr1e-2_seed_{seed}", "batch_size":64, "lr":1e-2, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_lr.append({"run_name":f"mbv2_bs64_lr1e-3_seed_{seed}", "batch_size":64, "lr":1e-3, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_lr.append({"run_name":f"mbv2_bs64_lr1e-4_seed_{seed}", "batch_size":64, "lr":1e-4, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})

# print(experiments_lr)

# results = []
# for cfg in experiments_lr:
#     run_dir, best_f1 = train_one_sample(cfg)
#     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

# Batch size experiments

In [ ]:
# experiments_bs = []

# for seed in SEEDS:
#     experiments_bs.append({"run_name":f"mbv2_bs32_lr1e-2_seed_{seed}", "batch_size":32, "lr":1e-2, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_bs.append({"run_name":f"mbv2_bs64_lr1e-2_seed_{seed}", "batch_size":64, "lr":1e-2, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_bs.append({"run_name":f"mbv2_bs128_lr1e-2_seed_{seed}", "batch_size":128, "lr":1e-2, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})

# print(experiments_bs)

# results = []
# for cfg in experiments_bs:
#     run_dir, best_f1 = train_one_sample(cfg)
#     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

# Weight decay experiments

In [ ]:
# experiments_bs = []

# for seed in SEEDS:
#     experiments_bs.append({"run_name":f"mbv2_bs32_lr1e-3_wd_1e-4_seed_{seed}", "batch_size":32, "lr":1e-3, "wd":1e-4, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_bs.append({"run_name":f"mbv2_bs32_lr1e-3_wd_1e-5_seed_{seed}", "batch_size":32, "lr":1e-3, "wd":1e-5, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     # experiments_bs.append({"run_name":f"mbv2_bs32_lr1e-3_seed_{seed}", "batch_size":128, "lr":1e-3, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})

# print(experiments_bs)

# results = []
# for cfg in experiments_bs:
#     run_dir, best_f1 = train_one_sample(cfg)
#     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

# Dropout rate experiments

In [ ]:
# experiments_drop = []

# for seed in SEEDS:
#     # experiments_drop.append({"run_name":f"mbv2_bs32_lr1e-3_wd_1e-4_dp_0_seed_{seed}", "batch_size":32, "lr":1e-3, "wd":1e-4, "dropout": 0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_drop.append({"run_name":f"mbv2_bs32_lr1e-3_wd_1e-4_dp_3e-1_seed_{seed}", "batch_size":32, "lr":1e-3, "wd":1e-4, "dropout": 0.3, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     # experiments_bs.append({"run_name":f"mbv2_bs32_lr1e-3_seed_{seed}", "batch_size":128, "lr":1e-3, "wd":0, "epochs":25, "ckpt_minutes":15, "seed": seed})

# print(experiments_drop)

# results = []
# for cfg in experiments_drop:
#     run_dir, best_f1 = train_one_sample(cfg)
#     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

# Augmentation experiments

In [ ]:
# experiments_aug = []

# for seed in SEEDS:
#     experiments_drop.append({"run_name":f"mbv2_bs32_lr1e-3_wd_1e-4_dp_2e-1_aug_none_seed_{seed}", "batch_size":32, "lr":1e-2, "wd":1e-4, "dropout": 0, "epochs":25, "ckpt_minutes":15, "seed": seed})
#     experiments_drop.append({"run_name":f"mbv2_bs32_lr1e-3_wd_1e-4_dp_3e-1_aug_flip_seed_{seed}", "batch_size":32, "lr":1e-2, "wd":1e-5, "dropout": 0.3, "epochs":25, "ckpt_minutes":15, "seed": seed})

# print(experiments_aug)

# # results = []
# # for cfg in experiments_drop:
# #     run_dir, best_f1 = train_one_sample(cfg)
# #     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

In [ ]:
# lr = 0.01
# bs = 128
# weight_decay = 0
# dropout = 0

# experiments_aug = []
# for seed in SEEDS:
#     experiments_aug.append({"run_name":f"mbv2_cutout_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutout"})
#     experiments_aug.append({"run_name":f"mbv2_cutmix1_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_1"})
#     experiments_aug.append({"run_name":f"mbv2_cutmix4_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_4"})

# print(experiments_aug)

# results = []
# for cfg in experiments_aug:
#     run_dir, best_f1 = train_one_sample(cfg)
#     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

In [ ]:
print(DATA_PATH)

/root/.cache/kagglehub/datasets/mengcius/cinic10/versions/1


In [ ]:
lr = 0.01
bs = 128
weight_decay = 0
dropout = 0

experiments_rand = []
for seed in SEEDS:
    # experiments_aug.append({"run_name":f"mbv2_cutout_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutout"})
    experiments_rand.append({"run_name":f"mbv2_random_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "random"})
    # experiments_aug.append({"run_name":f"mbv2_cutmix4_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_4"})

print(experiments_rand)

results = []
for cfg in experiments_rand:
    run_dir, best_f1 = train_one_sample(cfg)
    results.append({"run_dir": run_dir, "best_val_f1": best_f1})

[{'run_name': 'mbv2_random_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'random'}, {'run_name': 'mbv2_random_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'random'}, {'run_name': 'mbv2_random_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 2024, 'aug': 'random'}, {'run_name': 'mbv2_random_seed_7', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 7, 'aug': 'random'}, {'run_name': 'mbv2_random_seed_999', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 999, 'aug': 'random'}]
Starting run: mbv2_random_seed_42 with config: {'run_name': 'mbv2_random_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minute

In [ ]:
lr = 0.01
bs = 128
weight_decay = 0
dropout = 0

experiments_aug = []
for seed in SEEDS:
    # experiments_aug.append({"run_name":f"mbv2_cutout_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutout"})
    experiments_aug.append({"run_name":f"mbv2_cutmix1_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_1"})
    experiments_aug.append({"run_name":f"mbv2_cutmix4_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_4"})

print(experiments_aug)

results = []
for cfg in experiments_aug:
    run_dir, best_f1 = train_one_sample(cfg)
    results.append({"run_dir": run_dir, "best_val_f1": best_f1})

[{'run_name': 'mbv2_cutmix1_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 2024, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_min

In [ ]:
# lr = 0.01
# bs = 128
# weight_decay = 0
# dropout = 0

# experiments_aug = []
# for seed in SEEDS:
#     # experiments_aug.append({"run_name":f"mbv2_cutout_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutout"})
#     experiments_aug.append({"run_name":f"mbv2_cutmix1_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_1"})
#     experiments_aug.append({"run_name":f"mbv2_cutmix4_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_4"})

# print(experiments_aug)

# results = []
# for cfg in experiments_aug:
#     run_dir, best_f1 = train_one_sample(cfg)
#     results.append({"run_dir": run_dir, "best_val_f1": best_f1})

[{'run_name': 'mbv2_cutmix1_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 2024, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_min

KeyboardInterrupt: 

In [ ]:
lr = 0.01
bs = 128
weight_decay = 0
dropout = 0

experiments_aug = []
for seed in SEEDS:
    # experiments_aug.append({"run_name":f"mbv2_cutout_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutout"})
    experiments_aug.append({"run_name":f"mbv2_cutmix1_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_1"})
    experiments_aug.append({"run_name":f"mbv2_cutmix4_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_4"})

print(experiments_aug)

results = []
for cfg in experiments_aug:
    run_dir, best_f1 = train_one_sample(cfg)
    results.append({"run_dir": run_dir, "best_val_f1": best_f1})

[{'run_name': 'mbv2_cutmix1_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 2024, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_min

KeyboardInterrupt: 

In [ ]:
lr = 0.01
bs = 128
weight_decay = 0
dropout = 0

experiments_aug = []
for seed in SEEDS:
    # experiments_aug.append({"run_name":f"mbv2_cutout_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutout"})
    experiments_aug.append({"run_name":f"mbv2_cutmix1_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_1"})
    experiments_aug.append({"run_name":f"mbv2_cutmix4_seed_{seed}", "batch_size":bs, "lr":lr, "weight_decay":weight_decay, "dropout": dropout, "epochs":25, "ckpt_minutes":15, "seed": seed, "aug": "cutmix_alpha_4"})

print(experiments_aug)

results = []
for cfg in experiments_aug:
    run_dir, best_f1 = train_one_sample(cfg)
    results.append({"run_dir": run_dir, "best_val_f1": best_f1})

[{'run_name': 'mbv2_cutmix1_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_42', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 42, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_123', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 123, 'aug': 'cutmix_alpha_4'}, {'run_name': 'mbv2_cutmix1_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_minutes': 15, 'seed': 2024, 'aug': 'cutmix_alpha_1'}, {'run_name': 'mbv2_cutmix4_seed_2024', 'batch_size': 128, 'lr': 0.01, 'weight_decay': 0, 'dropout': 0, 'epochs': 25, 'ckpt_min